# Shape Classification with MMM (MedicalMultiTaskModeling)

Uses the `NativeBlocks` MMM encoder to classify shape images against a `labels.csv` ground-truth file.

**Requires Python 3.11 and the `mmm` conda env.**

**Expected `labels.csv` format:**
```
image_name,label
circle_001.png,circle
square_042.png,square
```

## 1. Install

In [1]:
!/opt/conda/envs/mmm/bin/pip install medicalmultitaskmodeling[interactive] numpy pandas matplotlib Pillow scikit-learn -q
print('Done')

Done


## 2. Imports

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

try:
    from mmm.labelstudio_ext.NativeBlocks import NativeBlocks, MMM_MODELS, DEFAULT_MODEL
    print("✅ MMM imported successfully")
    print(f"Available models: {list(MMM_MODELS.keys())}")
    print(f"Default model: {DEFAULT_MODEL}")
except ImportError as e:
    print(f"❌ MMM import failed: {e}")

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")


❌ MMM import failed: No module named 'mmm'
PyTorch: 2.10.0+cu130
CUDA: True
Device: cuda


## 3. Configuration

In [4]:
# ── Edit these ────────────────────────────────────────────────
IMAGE_DIR   = "./shapes_data"
LABELS_CSV  = "./labels.csv"
OUTPUT_CSV  = "./predictions.csv"

SHAPE_CLASSES = [
    "circle", "square", "triangle", "rectangle",
    "pentagon", "hexagon", "star", "ellipse",
]
# ──────────────────────────────────────────────────────────────

NUM_CLASSES = len(SHAPE_CLASSES)
IMAGE_SIZE  = 224
BATCH_SIZE  = 16

image_dir  = Path(IMAGE_DIR)
labels_csv = Path(LABELS_CSV)

assert image_dir.exists(),  f"Image directory not found: {image_dir}"
assert labels_csv.exists(), f"Labels CSV not found: {labels_csv}"
print(f"Image dir : {image_dir.resolve()}")
print(f"Labels CSV: {labels_csv.resolve()}")


AssertionError: Image directory not found: shapes_data

## 4. Load Labels CSV

In [ ]:
labels_df = pd.read_csv(labels_csv)
labels_df.columns = [c.strip().lower() for c in labels_df.columns]

name_col  = next((c for c in labels_df.columns if any(x in c for x in ['image','file','name'])), labels_df.columns[0])
label_col = next((c for c in labels_df.columns if any(x in c for x in ['label','class'])),       labels_df.columns[1])

labels_df = labels_df.rename(columns={name_col: 'image_name', label_col: 'label'})
label_map = dict(zip(labels_df['image_name'], labels_df['label']))

print(f"Loaded {len(label_map)} entries | classes: {sorted(labels_df['label'].unique())}")
labels_df.head()


## 5. Discover Images

In [ ]:
SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}
image_paths = sorted([p for p in image_dir.iterdir() if p.suffix.lower() in SUPPORTED_EXTS])
print(f"Found {len(image_paths)} images")

missing = [p.name for p in image_paths if p.name not in label_map]
if missing:
    print(f"⚠️  {len(missing)} images not in labels.csv — will still be classified")


## 6. Load MMM Model

In [ ]:
# Downloads automatically on first run (~few GB)
model    = NativeBlocks(MMM_MODELS[DEFAULT_MODEL], device_identifier=str(DEVICE))
encoder  = model["encoder"]
squeezer = model["squeezer"]

# Sanity check + get embedding dim
with torch.inference_mode():
    dummy          = torch.rand(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(model.device)
    feature_pyramid = encoder(dummy)
    hidden         = nn.Flatten(1)(squeezer(feature_pyramid)[1])

EMBED_DIM = hidden.shape[-1]
print(f"✅ Encoder ready | embedding dim: {EMBED_DIM}")

# Linear classification head (random weights — fine-tune or load checkpoint)
clf_head = nn.Linear(EMBED_DIM, NUM_CLASSES).to(model.device)
# clf_head.load_state_dict(torch.load('clf_head.pth'))  # ← load fine-tuned weights here
clf_head.eval()
print(f"Classification head: {EMBED_DIM} → {NUM_CLASSES}")


## 7. Preprocessing Pipeline

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def load_image(path: Path) -> torch.Tensor:
    return preprocess(Image.open(path).convert('RGB'))

print("Preprocessing pipeline ready.")


## 8. Run Inference

In [ ]:
results = []

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

print(f"Running inference on {len(image_paths)} images...")

with torch.inference_mode():
    for batch_paths in chunked(image_paths, BATCH_SIZE):
        tensors, valid_paths = [], []
        for p in batch_paths:
            try:
                tensors.append(load_image(p)); valid_paths.append(p)
            except Exception as e:
                print(f"  ⚠️  {p.name}: {e}")
        if not tensors:
            continue

        batch_tensor    = torch.stack(tensors).to(model.device)
        feature_pyramid = encoder(batch_tensor)
        hidden          = nn.Flatten(1)(squeezer(feature_pyramid)[1])
        logits          = clf_head(hidden)
        probs           = F.softmax(logits, dim=-1).cpu().numpy()

        for path, prob_vec in zip(valid_paths, probs):
            pred_idx   = int(np.argmax(prob_vec))
            pred_label = SHAPE_CLASSES[pred_idx]
            confidence = float(prob_vec[pred_idx])
            gt_label   = label_map.get(path.name, 'N/A')
            correct    = (pred_label == gt_label) if gt_label != 'N/A' else None
            results.append({
                'image_name': path.name, 'ground_truth': gt_label,
                'prediction': pred_label, 'confidence': round(confidence, 4), 'correct': correct,
                **{f'p_{cls}': round(float(p), 4) for cls, p in zip(SHAPE_CLASSES, prob_vec)},
            })

results_df = pd.DataFrame(results)
print(f"\n✅ Done. {len(results_df)} images processed.")
results_df.head(10)


## 9. Accuracy Metrics

In [ ]:
eval_df = results_df[results_df['ground_truth'] != 'N/A'].copy()

if len(eval_df) == 0:
    print("No labelled images found.")
else:
    print(f"Overall accuracy: {eval_df['correct'].mean():.2%}  ({eval_df['correct'].sum()}/{len(eval_df)})")
    per_class = (eval_df.groupby('ground_truth')['correct']
                 .agg(['sum','count'])
                 .rename(columns={'sum':'correct','count':'total'}))
    per_class['accuracy'] = per_class['correct'] / per_class['total']
    print("\nPer-class:\n", per_class.sort_values('accuracy').to_string())


## 10. Confusion Matrix

In [ ]:
if len(eval_df) > 0:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    classes = sorted(eval_df['ground_truth'].unique())
    cm = confusion_matrix(eval_df['ground_truth'], eval_df['prediction'], labels=classes)
    fig, ax = plt.subplots(figsize=(max(6, len(classes)), max(5, len(classes))))
    ConfusionMatrixDisplay(cm, display_labels=classes).plot(ax=ax, cmap='Blues')
    ax.set_title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()


## 11. Sample Predictions

In [ ]:
N_SHOW = min(16, len(results_df))
sample = results_df.sample(N_SHOW, random_state=42).reset_index(drop=True)
cols, rows = 4, (N_SHOW + 3) // 4
fig, axes = plt.subplots(rows, cols, figsize=(4*cols, 4*rows))
axes = axes.flatten()

for i, row in sample.iterrows():
    try:
        axes[i].imshow(mpimg.imread(image_dir / row['image_name']))
    except Exception:
        axes[i].text(0.5, 0.5, 'error', ha='center', va='center')
    colour = 'green' if row['correct'] else ('red' if row['correct'] is False else 'orange')
    axes[i].set_title(f"GT: {row['ground_truth']}\nPred: {row['prediction']} ({row['confidence']:.0%})", color=colour, fontsize=9)
    axes[i].axis('off')

for j in range(i+1, len(axes)): axes[j].axis('off')
plt.suptitle('Sample Predictions (green=correct, red=wrong)', y=1.01)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()


## 12. Save Results

In [ ]:
results_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Saved to {OUTPUT_CSV}")
results_df.tail(5)
